# 📚 livros-converter — Colab Free (T4 GPU)

Pipeline OCR + LLM para converter livros didáticos em Markdown estruturado para RAG.

**Antes de começar:**
1. Ative GPU: `Editar → Configurações do notebook → T4 GPU`
2. Tenha sua **chave Gemini** pronta: https://aistudio.google.com/apikey
3. Tenha o(s) PDF(s) em `MyDrive/livros-input/`

**Tempo estimado para 1 capítulo (~26 págs):** ~12 min

## 1. Verificar GPU

In [1]:
!nvidia-smi | head -20
import torch
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available(),
      'device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

Sun Apr 26 05:19:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Montar Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/livros-input', exist_ok=True)
os.makedirs('/content/drive/MyDrive/livros-output', exist_ok=True)
os.makedirs('/content/drive/MyDrive/hf_cache', exist_ok=True)

# Linka cache HF pro Drive (evita rebaixar GLM-OCR a cada sessão)
!mkdir -p /root/.cache && rm -rf /root/.cache/huggingface
!ln -sf /content/drive/MyDrive/hf_cache /root/.cache/huggingface

print('OK')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OK


## 3. Clonar repositório

Repo público — sem necessidade de autenticação.

In [ ]:
%cd /content
import os
if not os.path.isdir('livros-converter'):
    !git clone https://github.com/Goncalui/livros-converter.git
%cd livros-converter
!git pull --ff-only
!ls

## 4. Instalar dependências (Node + Python)

~5 min na primeira vez (baixa torch + paddle + transformers).

In [9]:
%%bash
set -e

# Node 22 (Colab vem com versão mais antiga)
if ! node -v 2>/dev/null | grep -q v22; then
  curl -fsSL https://deb.nodesource.com/setup_22.x | bash - >/dev/null 2>&1
  apt-get install -y nodejs >/dev/null 2>&1
fi
node --version

# Deps Node
npm install --silent

# Deps Python
pip install -q PyMuPDF Pillow pytesseract \
  paddlepaddle==3.0.0 paddleocr==3.5.0 \
  glmocr 'transformers>=5.0' accelerate sentencepiece pypdfium2 opencv-python-headless

# Tesseract com pt
apt-get install -y -q tesseract-ocr tesseract-ocr-por
echo 'tesseract:'; tesseract --version 2>&1 | head -1
echo 'OK — deps instaladas'

v22.22.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 9.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.9/192.9 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.8/120.8 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.2/115.2 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 106.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.


## 5. Configurar `.env` com chave Gemini

**Substitua** `COLE_AQUI_SUA_CHAVE` pela sua chave real.

In [ ]:
from getpass import getpass

# Cole sua chave Gemini quando pedir (entrada oculta)
GEMINI_API_KEY = getpass('Chave Gemini API (entrada oculta): ').strip()
assert GEMINI_API_KEY, 'chave vazia!'

env = f'''# OCR — só GLM-OCR
OCR_MODE=glm
OCR_MIN_CONFIDENCE=0.80

# GLM-OCR (transformers direct, GPU)
GLM_OCR_MODE=selfhosted
GLM_OCR_DTYPE=fp16
GLM_OCR_BATCH_SIZE=2
GLM_OCR_MAX_NEW_TOKENS=4096
CUDA_VISIBLE_DEVICES=0
GLM_LAYOUT_DEVICE=cuda

# LLM — Gemini como primário (Claude CLI não roda no Colab)
LLM_ORDER_TEXT=gemini,ollama
LLM_ORDER_VISION=gemini,ollama
LLM_VISION_ESCALATION=0
LLM_STREAMING=1
LLM_MAX_INFLIGHT=1
GEMINI_API_KEY={GEMINI_API_KEY}
GEMINI_TEXT_MODEL=gemini-3.1-flash-lite-preview
GEMINI_VISION_MODEL=gemini-3.1-flash-lite-preview

# Pipeline — lote de 5 págs com 1 de overlap (LLM dispara assim que 5 ficam prontas)
BATCH_SIZE=5
BATCH_OVERLAP=1
PYTHON_BIN=python3
'''

with open('/content/livros-converter/.env', 'w') as f:
    f.write(env)

print('.env criado com LLM_STREAMING=1. Chave: ' + GEMINI_API_KEY[:6] + '…')
del GEMINI_API_KEY

## 6. Recortar capítulo do PDF

Edite `PDF`, `PAGE_FROM` e `PAGE_TO` conforme seu livro.
Rode primeiro com o sumário aberto (último print) pra escolher o intervalo.

In [8]:
import fitz, os

PDF      = '/content/drive/MyDrive/livros-input/Biologia  - Volume 1.pdf'
OUT_DIR  = '/content/livros-converter/input-bio-completo'

doc = fitz.open(PDF)
total = len(doc)
print(f'PDF tem {total} páginas. Renderizando todas...')

os.makedirs(OUT_DIR, exist_ok=True)
for i in range(total):
    pix = doc[i].get_pixmap(dpi=200)
    pix.save(f'{OUT_DIR}/page-{i+1:03d}.png')
    if (i+1) % 25 == 0:
        print(f'  …{i+1}/{total}')
print(f'OK — {total} páginas em {OUT_DIR}')

PDF tem 367 páginas. Renderizando todas...
  …25/367
  …50/367
  …75/367
  …100/367
  …125/367
  …150/367
  …175/367
  …200/367
  …225/367
  …250/367
  …275/367
  …300/367
  …325/367
  …350/367
OK — 367 páginas em /content/livros-converter/input-bio-completo


In [3]:

!pip install -q -U 'transformers>=5.6' accelerate

# valida
import transformers, importlib
importlib.reload(transformers)
print('transformers:', transformers.__version__)
try:
    from transformers.models.glm_ocr import GlmOcrForConditionalGeneration
    print('✓ GlmOcrForConditionalGeneration disponível')
except Exception as e:
    print('✗', e)

transformers: 5.6.2
✓ GlmOcrForConditionalGeneration disponível


In [6]:
!cd /content/livros-converter && npm install --silent && mkdir -p workspace
!ls /content/livros-converter/node_modules | head -5

agent-base
base64-js
bignumber.js
buffer-equal-constant-time
debug


## 8. Rodar pipeline

Saída em tempo real. Estimativa T4: OCR ~8min · Convert Gemini ~3min · Total ~12min.

Se travar/desconectar, rode `node src/cli.js resume input-bio-cap1` numa nova célula — retoma do checkpoint.

In [ ]:
!mkdir -p /content/drive/MyDrive/livros-output
!cd /content/livros-converter && \
  WORKSPACE_ROOT=/content/drive/MyDrive/livros-output \
  PYTHONIOENCODING=utf-8 \
  node src/cli.js convert ./input-bio-completo 2>&1 | tee /content/drive/MyDrive/livros-output/run.log

## 9. Conferir output no Drive

Workspace já é gravado incrementalmente em `MyDrive/livros-output/<slug>/` com fsync por página/lote — mesmo que o Colab caia, o progresso está salvo.

In [ ]:
%%bash
SLUG=input-bio-completo
DST=/content/drive/MyDrive/livros-output/$SLUG/output
echo "Output em: $DST"
ls -la "$DST" 2>/dev/null || echo "(ainda não gerado)" 

## 10. (Opcional) Mostrar markdown gerado

In [ ]:
from IPython.display import Markdown, display
import glob, os

out_dir = '/content/drive/MyDrive/livros-output/input-bio-completo/output'
files = sorted(glob.glob(f'{out_dir}/cap-*.md'))
print(f'{len(files)} capítulos gerados:')
for f in files:
    print(' -', os.path.basename(f), f'({os.path.getsize(f)//1024} KB)')

if files:
    print('\n=== Preview do primeiro capítulo (3000 chars) ===')
    with open(files[0], 'r', encoding='utf-8') as f:
        content = f.read()
    display(Markdown(content[:3000] + '\n\n…'))

## 11. (Opcional) Comparação OCR — auditar Paddle vs GLM vs Tesseract

Mostra qual engine ganhou em cada página e métricas brutas.

In [ ]:
import json, glob
from collections import Counter

raw = '/content/drive/MyDrive/livros-output/input-bio-completo/raw'
files = sorted(glob.glob(f'{raw}/page-*.compare.json'))
winners = Counter()
rows = []
for f in files:
    j = json.load(open(f))
    winners[j.get('winner') or 'none'] += 1
    eng = j.get('engines', {})
    rows.append({
        'pág': j['page'],
        'vencedor': j.get('winner'),
        **{f'{k}_score': eng.get(k, {}).get('score') for k in ['paddle','glm','tesseract']}
    })

print('=== Vencedores ===')
for k, v in winners.most_common():
    print(f'  {k}: {v}')

if rows:
    try:
        import pandas as pd
        print('\n=== Tabela ==='); print(pd.DataFrame(rows).to_string(index=False))
    except ImportError:
        for r in rows: print(r)
else:
    print('(modo glm puro — sem comparação. Mude OCR_MODE=compare para gerar.)')

---

## 🎯 Pronto!

Resultado salvo **incrementalmente** em `MyDrive/livros-output/input-bio-completo/`:
- `output/index.md` — sumário
- `output/cap-NN-*.md` — markdown estruturado por capítulo
- `output/_full.md` — versão concatenada
- `output/_validation.json` — avisos de validação
- `raw/page-NNN.txt` — saída do OCR por página
- `.state.json` — checkpoint do pipeline

**Para rodar outro livro/capítulo:** ajuste `PDF` e `OUT_DIR` na Célula 12, depois rode 12 → 16. Mude também o slug nas células 18, 20 e 22.

**Para retomar após desconexão:**
```bash
!cd /content/livros-converter && WORKSPACE_ROOT=/content/drive/MyDrive/livros-output node src/cli.js resume input-bio-completo
```
O OCR pula páginas já processadas (filtra por `text_source` no manifest) e o conversor pula lotes já feitos (lista `batchesDone` no `.state.json`).